# seaborn으로 빠르게

> 파이썬 11강 · 시각화

이 노트북은 웹 강의의 **실습 부분만** 옮겨온 것입니다.
자세한 설명과 그림은 원문을 함께 보세요 → [seaborn으로 빠르게](https://mioon1402.github.io/timeseriesdata/python/p11-seaborn.html)

---

**먼저 아래 준비 셀을 한 번 실행하세요.** 예시 데이터를 내려받습니다.

In [ ]:
# 예시 데이터 내려받기
!wget -q -nc https://raw.githubusercontent.com/mioon1402/timeseriesdata/main/data/cafe_sales.csv

# Colab 에는 대부분 설치돼 있지만, 없으면 아래 주석을 푸세요
# !pip install -q seaborn

# 그래프 한글 깨짐 방지
!pip install -q koreanize-matplotlib
import koreanize_matplotlib  # noqa: F401

print('준비 완료')

## 1. 한 줄의 차이

**11-1. matplotlib vs seaborn**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("cafe_sales.csv", parse_dates=["date"])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.4))

# matplotlib — 데이터를 직접 넘긴다
ax1.hist(df["visitors"], bins=30, edgecolor="white")
ax1.set_title("matplotlib")

# seaborn — 데이터프레임과 '열 이름'을 넘긴다
sns.histplot(data=df, x="visitors", bins=30, ax=ax2)
ax2.set_title("seaborn")

plt.tight_layout()
plt.show()

## 2. 분포 보기

**11-2. 히스토그램과 밀도곡선**

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))

sns.histplot(data=df, x="visitors", bins=30, ax=axes[0])
axes[0].set_title("histplot — 도수")

sns.histplot(data=df, x="visitors", bins=30, kde=True, ax=axes[1])
axes[1].set_title("kde=True — 매끄러운 곡선 겹치기")

sns.kdeplot(data=df, x="visitors", fill=True, ax=axes[2])
axes[2].set_title("kdeplot — 곡선만")

plt.tight_layout()
plt.show()

## 3. 범주 비교

**11-3. 상자그림 · 바이올린 · 막대**

In [ ]:
순서 = ["월", "화", "수", "목", "금", "토", "일"]

fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))

sns.boxplot(data=df, x="weekday", y="visitors", order=순서, ax=axes[0])
axes[0].set_title("boxplot — 다섯 숫자 요약")

sns.violinplot(data=df, x="weekday", y="visitors", order=순서, ax=axes[1])
axes[1].set_title("violinplot — 분포 모양까지")

sns.barplot(data=df, x="weekday", y="visitors", order=순서, ax=axes[2])
axes[2].set_title("barplot — 평균 + 신뢰구간")

plt.tight_layout()
plt.show()

**11-4. 상자그림 위에 점 겹치기**

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4))

sns.boxplot(data=df, x="weekday", y="visitors", order=순서,
            showfliers=False, ax=ax, color="#e2e8f0")
sns.stripplot(data=df, x="weekday", y="visitors", order=순서,
              size=2.5, alpha=0.4, color="#2563eb", ax=ax)

ax.set_title("요일별 방문객 — 요약과 원본을 함께")
ax.set_xlabel(""); ax.set_ylabel("방문객(명)")
plt.show()

## 4. 관계 보기

**11-5. 산점도와 회귀선**

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.8))

sns.scatterplot(data=df, x="avg_temp", y="sales", alpha=0.4, s=18, ax=ax1)
ax1.set_title("scatterplot")

# regplot — 산점도 + 회귀선 + 신뢰밴드를 한 번에
sns.regplot(data=df, x="avg_temp", y="sales",
            scatter_kws={"alpha": 0.25, "s": 12},
            line_kws={"color": "#dc2626"}, ax=ax2)
ax2.set_title("regplot — 회귀선까지")

plt.tight_layout()
plt.show()

## 5. 상관 히트맵

**11-6. 한눈에 보는 상관관계**

In [ ]:
수치열 = df[["visitors", "sales", "avg_temp", "rain_mm"]]
상관 = 수치열.corr()

fig, ax = plt.subplots(figsize=(5.5, 4.5))
sns.heatmap(상관, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, square=True, ax=ax)
ax.set_title("변수 간 상관계수")
plt.tight_layout()
plt.show()

## 6. hue — 세 번째 변수를 색으로

**11-7. hue 로 그룹 나누기**

In [ ]:
df["비"] = df["rain_mm"] > 0

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 3.8))

sns.scatterplot(data=df, x="avg_temp", y="sales", hue="비",
                alpha=0.45, s=18, ax=ax1)
ax1.set_title("비 여부로 색 나누기")

sns.boxplot(data=df, x="weekday", y="visitors", order=순서,
            hue="비", ax=ax2)
ax2.set_title("요일 × 비 여부")

plt.tight_layout()
plt.show()

**11-8. pairplot — 전체를 한 번에**

In [ ]:
# 변수가 몇 개 안 될 때, 모든 조합을 한 번에 훑는다
작은표본 = df[["visitors", "avg_temp", "rain_mm", "비"]].sample(300, random_state=42)

g = sns.pairplot(작은표본, hue="비", height=1.9,
                 plot_kws={"alpha": 0.5, "s": 12})
plt.show()

## 7. 언제 matplotlib으로 돌아가나

**11-9. 섞어서 완성본 만들기**

In [ ]:
sns.set_theme(style="whitegrid")     # seaborn 의 기본 스타일

fig, ax = plt.subplots(figsize=(9, 4))
sns.boxplot(data=df, x="weekday", y="visitors", order=순서,
            ax=ax, color="#93c5fd", showfliers=False)

# 여기서부터는 matplotlib 문법
ax.set_title("주말로 갈수록 손님이 늘고, 편차도 커진다",
             fontsize=13, fontweight="bold", loc="left", pad=12)
ax.set_xlabel(""); ax.set_ylabel("방문객(명)")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

**연습 · 직접 써보세요**

In [ ]:
# 문제 1. 계절별 매출 분포를 boxplot 으로 그려보세요.
#        힌트: 월을 계절로 바꾸는 함수는 4강에서 만들었습니다


# 문제 2. 기온과 방문객의 regplot 을 그리되,
#        주말/평일을 hue 로 구분해보세요.
#        힌트: regplot 에는 hue 가 없습니다. lmplot 을 쓰세요


# 문제 3. 상관 히트맵에 '주말' 열을 추가해서 다시 그려보세요.

**모범 답안**

In [ ]:
def 계절(m):
    if m in (3, 4, 5):   return "봄"
    if m in (6, 7, 8):   return "여름"
    if m in (9, 10, 11): return "가을"
    return "겨울"

df["계절"] = df["date"].dt.month.map(계절)
df["주말"] = df["date"].dt.dayofweek >= 5

# 문제 1
fig, ax = plt.subplots(figsize=(7, 3.6))
sns.boxplot(data=df, x="계절", y="sales",
            order=["봄", "여름", "가을", "겨울"], ax=ax)
ax.set_title("계절별 매출 분포", fontweight="bold", loc="left")
plt.show()

# 문제 2 — hue 가 필요하면 lmplot (figure 를 직접 만든다)
sns.lmplot(data=df, x="avg_temp", y="visitors", hue="주말",
           height=4, aspect=1.5, scatter_kws={"alpha": 0.3, "s": 12})
plt.show()

# 문제 3
상관2 = df[["visitors", "sales", "avg_temp", "rain_mm", "주말"]].corr()
fig, ax = plt.subplots(figsize=(6, 4.8))
sns.heatmap(상관2, annot=True, fmt=".2f", cmap="RdBu_r",
            center=0, vmin=-1, vmax=1, square=True, ax=ax)
plt.tight_layout()
plt.show()

---

전체 강의 목록 → [눈으로 보는 통계](https://mioon1402.github.io/timeseriesdata/)